In [2]:
import tensorflow as tf

In [3]:
from tensorflow.keras.layers import Input,Concatenate

In [4]:
from tensorflow.keras.models import Model

# Features

In [5]:
features={
  "NUMERICAL_FEATURES": [
    "num_single_quote_error",
    "num_spacing_error",
    "num_space_absence_after_sentence_completion",
    "num_capitalized_words",
    "num_capitalization_absence_after_sentence_completion",
    "num_spelling_errors",
    "num_punctuations",
    "num_numeric_values",
    "num_words"
  ],

  "TEXT_FEATURE": "text",
  "LABEL": "label"
}

# Numerical Feature Processing Network

In [6]:
from models.model_for_numerical_feature.numerical_feature_processing_layer import NumericalFeatureProcessingLayer

In [7]:
model_inp=[]
numerical_features=features["NUMERICAL_FEATURES"]
for feat in numerical_features:
    inp=Input(
        shape=(1,),
        dtype=tf.float32,
        name=feat
    )
    model_inp.append(inp)

In [8]:
dense_layer_config=[
    {
        "units":16,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None
    },
    {
        "units":32,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None
    }
]
  

In [9]:
out=Concatenate(axis=-1)(model_inp)
out=NumericalFeatureProcessingLayer(dense_layer_config=dense_layer_config)(out)

E0000 00:00:1782863553.331635  332850 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [10]:
model=Model(inputs=model_inp,outputs=out)

In [11]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ num_single_quote_e… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_spacing_error   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_space_absence_… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_capitalized_wo… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_capitalization… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_spelling_errors │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_punctuations    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_numeric_values  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_words           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 9)         │          0 │ num_single_quote… │
│ (Concatenate)       │                   │            │ num_spacing_erro… │
│                     │                   │            │ num_space_absenc… │
│                     │                   │            │ num_capitalized_… │
│                     │                   │            │ num_capitalizati… │
│                     │                   │            │ num_spelling_err… │
│                     │                   │            │ num_punctuations… │
│                     │                   │            │ num_numeric_valu… │
│                     │                   │            │ num_words[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numerical_feature_… │ (None, 32)        │        704 │ concatenate[0][0] │
│ (NumericalFeatureP… │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 704 (2.75 KB)

 Trainable params: 704 (2.75 KB)

 Non-trainable params: 0 (0.00 B)

# Text Processing Network

## Basic Text Processing Layer

In [12]:
from models.model_for_text_feature.basic_text_processing_layer import BasicTextProcessingLayer

In [13]:
lstm_layer_config=[
      {
        "units":4,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None,
        "return_sequences": True,
        "bidirectional": True
      },
      {
        "units":8,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None,
        "return_sequences": True,
        "bidirectional": False
      },
      {
        "units":16,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None,
        "return_sequences": False,
        "bidirectional":False
      }
    ]

In [14]:
dense_layer_config=[
      {
        "units":32,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None
      },
      {
        "units":64,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None
      }
    ]

In [15]:
embedding_dim=64
seq_len=60

In [16]:
model_inp=Input(shape=(seq_len,embedding_dim),dtype=tf.float32,name="embedded_text")

In [17]:
out=BasicTextProcessingLayer(
    lstm_layer_config=lstm_layer_config,
    dense_layer_config=dense_layer_config
)(model_inp)

In [18]:
model=Model(inputs=model_inp,outputs=out)

In [19]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedded_text (InputLayer)      │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ basic_text_processing_layer     │ (None, 64)             │         7,008 │
│ (BasicTextProcessingLayer)      │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,008 (27.38 KB)

 Trainable params: 7,008 (27.38 KB)

 Non-trainable params: 0 (0.00 B)

## Input Divider Layer

In [20]:
from models.model_for_text_feature.input_divider_layer import InputDividerLayer

In [21]:
num_models=6
embedding_dim=64
text_seq_len=360

In [22]:
model_inp=Input(shape=(text_seq_len,embedding_dim),dtype=tf.float32,name="embedded_text")

In [23]:
out=InputDividerLayer(num_models=num_models)(model_inp)

In [24]:
model=Model(inputs=model_inp,outputs=out)

In [25]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedded_text (InputLayer)      │ (None, 360, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ input_divider_layer             │ [(None, 60, 64),       │             0 │
│ (InputDividerLayer)             │ (None, 60, 64), (None, │               │
│                                 │ 60, 64), (None, 60,    │               │
│                                 │ 64), (None, 60, 64),   │               │
│                                 │ (None, 60, 64)]        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Multi LSTM Layer

In [26]:
from models.model_for_text_feature.multi_lstm_layer import MultiLSTMLayer

In [27]:
basic_text_processing_layer_config={
    "lstm_layer_config": [
      {
        "units":4,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None,
        "return_sequences": True,
        "bidirectional": True
      },
      {
        "units":8,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None,
        "return_sequences": True,
        "bidirectional": False
      },
      {
        "units":16,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None,
        "return_sequences": False,
        "bidirectional":False
      }
    ],
    "dense_layer_config":[
     {
        "units":32,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None
      },
      {
        "units":64,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None
      }
    ]
  }


In [28]:
dense_layer_config=[
    {
        "units":256,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None
    },
    {
        "units":128,
        "activation": "relu",
        "kernel_initializer": "he_uniform",
        "kernel_regularizer": None
    }
]

In [29]:
text_seq_len=360
num_models=6
embedding_dim=64

In [30]:
model_inps=[]
for ind in range(num_models):
    inp=Input(
        shape=(int(text_seq_len/num_models),embedding_dim),
        dtype=tf.float32,
        name=f"embedded_input_{ind+1}"
    )
    model_inps.append(inp)

In [31]:
out=MultiLSTMLayer(
    num_models=num_models,
    basic_text_processing_layer_config=basic_text_processing_layer_config,
    dense_layer_config=dense_layer_config
)(model_inps)

In [32]:
model=Model(inputs=model_inps,outputs=out)

In [33]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ embedded_input_1    │ (None, 60, 64)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedded_input_2    │ (None, 60, 64)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedded_input_3    │ (None, 60, 64)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedded_input_4    │ (None, 60, 64)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedded_input_5    │ (None, 60, 64)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedded_input_6    │ (None, 60, 64)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_lstm_layer    │ (None, 128)       │    173,504 │ embedded_input_1… │
│ (MultiLSTMLayer)    │                   │            │ embedded_input_2… │
│                     │                   │            │ embedded_input_3… │
│                     │                   │            │ embedded_input_4… │
│                     │                   │            │ embedded_input_5… │
│                     │                   │            │ embedded_input_6… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 173,504 (677.75 KB)

 Trainable params: 173,504 (677.75 KB)

 Non-trainable params: 0 (0.00 B)

## Text Processing Network

In [34]:
from models.model_for_text_feature.text_processing_network import TextProcessingLayer

In [35]:
num_models=6
embedding_dim=64
vocab_size=20000

In [36]:
model_inp=Input(
    shape=(360,),
    dtype=tf.int32,
    name="tokenized_text"

)

In [37]:
out=TextProcessingLayer(
    num_models=num_models,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    basic_text_processing_layer_config=basic_text_processing_layer_config,
    multi_lstm_dense_layer_config=dense_layer_config,
)(model_inp)

In [38]:
model=Model(inputs=model_inp,outputs=out)

In [39]:
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tokenized_text (InputLayer)     │ (None, 360)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_processing_layer           │ (None, 768)            │     1,453,504 │
│ (TextProcessingLayer)           │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,453,504 (5.54 MB)

 Trainable params: 1,453,504 (5.54 MB)

 Non-trainable params: 0 (0.00 B)